# The Evolution of Text Generation in Language Models

Code tests for the survey. Each cell is independent: run them top to bottom, or any one on its own.

Only needs `pip install numpy` (test 11b also needs `transformers` and `torch`).

## 3. Rule-Based Chatbots: ELIZA (1966)

**Example:** I am sad. → **Why are you sad?**

In [1]:
import re

# Hand-written rules: (pattern, reply template)
rules = [
    (r"i am (.*)",   "Why are you {0}?"),
    (r"i feel (.*)", "How long have you felt {0}?"),
    (r".*mother.*",  "Tell me more about your family."),
]

def eliza(text):
    text = text.lower().strip(".!? ")
    for pattern, reply in rules:
        match = re.match(pattern, text)
        if match:
            return reply.format(*match.groups())
    return "Please tell me more."      # no rule matched

for sentence in ["I am sad.", "I feel tired.", "The cat sat on the mat."]:
    print(sentence, "->", eliza(sentence))

I am sad. -> Why are you sad?
I feel tired. -> How long have you felt tired?
The cat sat on the mat. -> Please tell me more.


## 4. Counting Words: N-gram Models (1948 idea, widely used 1980s-2000s)

**Example:** The cat sat on the ___ → **mat (or cat, dog, rug, fish)**

In [2]:
from collections import Counter, defaultdict

corpus = "the cat sat on the mat . the dog sat on the rug . the cat ate the fish ."
words = corpus.split()

# Count which word comes after which word (bigram counts)
next_words = defaultdict(Counter)
for a, b in zip(words, words[1:]):
    next_words[a][b] += 1

def probabilities(prev):
    total = sum(next_words[prev].values())
    return {w: round(c / total, 2) for w, c in next_words[prev].items()}

print("after 'the' :", probabilities("the"))
print("after 'sat' :", probabilities("sat"))
print("after 'bird':", probabilities("bird"))   # never seen -> nothing

# Generate: always pick the most common next word
word, output = "the", ["the"]
for _ in range(7):
    word = next_words[word].most_common(1)[0][0]
    output.append(word)
print("generated   :", " ".join(output))

after 'the' : {'cat': 0.33, 'mat': 0.17, 'dog': 0.17, 'rug': 0.17, 'fish': 0.17}
after 'sat' : {'on': 1.0}
after 'bird': {}
generated   : the cat sat on the cat sat on


## 5. Neural Language Models and Word Vectors (2003, 2013)

**Example:** The dog slept ___   (training text only had "the cat slept") → **on the mat**

In [3]:
import numpy as np
np.random.seed(0)

# Training sentences. "the dog slept" is NEVER seen, but "the cat slept" is.
sentences = ["the cat sat on the mat", "the dog sat on the mat",
             "the cat ran to the door", "the dog ran to the door",
             "the cat slept on the sofa"]
vocab = sorted(set(" ".join(sentences).split()))
idx = {w: i for i, w in enumerate(vocab)}
V, D = len(vocab), 8                       # vocab size, vector size

# Training pairs: (two previous words) -> next word
X, Y = [], []
for s in sentences:
    w = s.split()
    for i in range(2, len(w)):
        X.append([idx[w[i-2]], idx[w[i-1]]]); Y.append(idx[w[i]])
X, Y = np.array(X), np.array(Y)

E = np.random.randn(V, D) * 0.1            # word vectors (embeddings)
W = np.random.randn(2 * D, V) * 0.1        # output layer

def predict(X):
    h = np.concatenate([E[X[:, 0]], E[X[:, 1]]], axis=1)
    z = h @ W
    p = np.exp(z - z.max(1, keepdims=True))
    return h, p / p.sum(1, keepdims=True)

for step in range(3000):                   # gradient descent
    h, p = predict(X)
    p[np.arange(len(Y)), Y] -= 1           # gradient of the loss
    gh = p @ W.T
    W -= 0.3 * (h.T @ p)
    np.add.at(E, X[:, 0], -0.3 * gh[:, :D])
    np.add.at(E, X[:, 1], -0.3 * gh[:, D:])

# Word vectors idea (Word2Vec): a word is described by its neighbours.
def neighbour_vector(word):
    v = np.zeros(V)
    for s in sentences:
        w = s.split()
        for i, x in enumerate(w):
            if x == word:
                for j in (i - 1, i + 1):
                    if 0 <= j < len(w) and w[j] != "the":   # skip the common word "the"
                        v[idx[w[j]]] += 1
    return v
cos = lambda a, b: a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print("similarity cat ~ dog :", round(cos(neighbour_vector("cat"), neighbour_vector("dog")), 2))
print("similarity cat ~ sat :", round(cos(neighbour_vector("cat"), neighbour_vector("sat")), 2))

# An n-gram model has ZERO counts for "dog slept" ...
print("times 'dog slept' seen in training:", " ".join(sentences).count("dog slept"))

# ... but the neural model can still continue it: "the dog slept ..."
words = ["the", "dog", "slept"]
for _ in range(3):
    _, p = predict(np.array([[idx[words[-2]], idx[words[-1]]]]))
    words.append(vocab[p[0].argmax()])
print("generated:", " ".join(words))

similarity cat ~ dog : 0.82
similarity cat ~ sat : 0.0
times 'dog slept' seen in training: 0
generated: the dog slept on the mat


## 6. Recurrent Neural Networks: RNN (1990, language models 2010)

**Example:** The cat that chased the dogs ___ tired. → **was   (singular, because of "cat", not "dogs")**

In [ ]:
import numpy as np
np.random.seed(1)

D = 8                                      # size of word vectors / memory
U = np.random.randn(D, D) * 0.3            # memory -> memory weights
Wx = np.random.randn(D, D)                 # word -> memory weights

def run_rnn(word_vectors):
    h = np.zeros(D)                        # empty memory at the start
    for x in word_vectors:                 # read ONE word at a time
        h = np.tanh(Wx @ x + U @ h)        # mix new word into memory
    return h

cat, dog = np.random.randn(D), np.random.randn(D)
filler = [np.random.randn(D) for _ in range(30)]   # words in between

# Does the FIRST word (cat vs dog) still change the final memory?
for gap in [1, 5, 10, 30]:
    h_cat = run_rnn([cat] + filler[:gap])
    h_dog = run_rnn([dog] + filler[:gap])
    print(f"{gap:2d} words later, difference = {np.abs(h_cat - h_dog).sum():.6f}")

## 7. Long Short-Term Memory: LSTM (1997, popular 2014-2016)

**Example:** The cat that chased the dogs ___ tired. → **was**

In [ ]:
import numpy as np
sigmoid = lambda z: 1 / (1 + np.exp(-z))

# One number of memory. At step 0 we store "cat" = 1.0, then 50 empty steps.
rnn_h = 0.0
lstm_c = 0.0
for t in range(51):
    x = 1.0 if t == 0 else 0.0
    # Simple RNN: memory is squashed and shrunk every step
    rnn_h = np.tanh(1.0 * x + 0.5 * rnn_h)
    # LSTM: gates decide what to keep, add and forget
    forget = sigmoid(5.0)                  # ~0.993 -> "keep almost all"
    add    = sigmoid(5.0 * x - 2.5)        # open only when a word arrives
    lstm_c = forget * lstm_c + add * x     # cell state = conveyor belt
    if t in (0, 5, 20, 50):
        print(f"step {t:2d}: RNN memory = {rnn_h:.4f}   LSTM memory = {lstm_c:.4f}")

## 8. Sequence-to-Sequence with Attention (2014)

**Example:** the cat sat  (English) → **le chat ...  (French)**

In [ ]:
import numpy as np

# Translate "the cat sat" -> French "le chat ..."
# Tiny hand-made word vectors: similar meanings point the same way.
english = {"the": [1, 0, 0], "cat": [0, 1, 0], "sat": [0, 0, 1]}
french_query = {"le": [0.9, 0.1, 0.0], "chat": [0.1, 0.9, 0.0]}

keys = np.array(list(english.values()), dtype=float)

def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()

for fr, q in french_query.items():
    scores = keys @ np.array(q) * 5          # how well each English word matches
    weights = softmax(scores)                # turn scores into percentages
    pretty = {w: f"{a:.0%}" for w, a in zip(english, weights)}
    print(f"writing '{fr}' looks at:", pretty)

## 9. The Transformer (2017)

**Example:** The cat sat on the ___ → **mat**

In [ ]:
import numpy as np
np.random.seed(3)

words = "the cat sat on the mat".split()
n, d = len(words), 4

X = np.random.randn(n, d)                       # one vector per word
X += np.array([[np.sin(p / 2), np.cos(p / 2), 0, 0] for p in range(n)])  # position info
Wq, Wk, Wv = (np.random.randn(d, d) * 0.5 for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv                # Query, Key, Value for ALL words at once

scores = Q @ K.T / np.sqrt(d)                   # every word vs every word
mask = np.triu(np.ones((n, n)), k=1) == 1       # GPT rule: no peeking at future words
scores[mask] = -np.inf
weights = np.exp(scores - scores.max(1, keepdims=True))
weights /= weights.sum(1, keepdims=True)        # softmax per row
output = weights @ V                            # new vector for every word

print("attention of 'sat' (row 3):")
for w, a in zip(words, weights[2]):
    print(f"  {w:4s} {a:.2f}")
print("output shape:", output.shape, "-> all 6 words processed together")

## 10. GPT-1: Pre-training on Plain Text (2018)

**Example:** the cat sat → **tokens [the] [cat] [sat] -> next token "on"**

In [ ]:
from collections import Counter

# GPT-style tokenizer idea (Byte-Pair Encoding):
# start with letters, repeatedly glue the most frequent neighbouring pair.
text = "the cat sat on the mat the cat ate"
words = [list(w) for w in text.split()]

for step in range(1, 6):
    pairs = Counter()
    for w in words:
        for a, b in zip(w, w[1:]):
            pairs[(a, b)] += 1
    (a, b), count = pairs.most_common(1)[0]
    merged = []
    for w in words:
        i, new = 0, []
        while i < len(w):
            if i < len(w) - 1 and (w[i], w[i + 1]) == (a, b):
                new.append(a + b); i += 2
            else:
                new.append(w[i]); i += 1
        merged.append(new)
    words = merged
    print(f"merge {step}: '{a}'+'{b}' -> '{a+b}'   tokens: {words[:3]}")

vocab = sorted({t for w in words for t in w})
ids = [[vocab.index(t) for t in w] for w in words[:3]]
print("token ids for 'the cat sat':", ids)

## 11. GPT-2: Bigger Models and Sampling (2019)

**Example:** The cat sat on the ___ → **mat 51%, sofa 20%, floor 20%, roof 8%, moon 1% (temperature 1.0)**

In [ ]:
import numpy as np
from collections import Counter
rng = np.random.default_rng(42)

# Pretend the model scored these next words after "The cat sat on the"
words  = ["mat", "sofa", "floor", "roof", "moon"]
scores = np.array([3.0, 2.2, 2.0, 1.0, -1.0])

def sample(temperature, top_k=None, n=1000):
    z = scores / temperature
    if top_k:                                  # keep only the k best words
        z = np.where(z >= np.sort(z)[-top_k], z, -np.inf)
    p = np.exp(z - z.max()); p /= p.sum()
    picks = Counter(words[i] for i in rng.choice(len(words), size=n, p=p))
    return {w: f"{picks[w] / n:.0%}" for w in words}

print("greedy (always the best):", words[scores.argmax()])
for t in [0.5, 1.0, 2.0]:
    print(f"temperature {t}:", sample(t))
print("top-k = 2      :", sample(1.0, top_k=2))

### 11b. Optional: real GPT-2

Uncomment the first line to install, then run (downloads about 550 MB once).

In [ ]:
# !pip install transformers torch
# OPTIONAL: real GPT-2 (needs: pip install transformers torch ; downloads ~550 MB once)
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
result = generator("The cat sat on the", max_new_tokens=12, do_sample=False)
print(result[0]["generated_text"])

## 12. GPT-3: Learning from Examples in the Prompt (2020)

**Example:** cat -> chat, dog -> chien, house -> maison, mat -> → **tapis**

In [ ]:
# GPT-3 idea: the task is taught INSIDE the prompt (no retraining).
# We only build the prompt here; paste it into any chat model to test.
examples = [("cat", "chat"), ("dog", "chien"), ("house", "maison")]
question = "mat"

zero_shot = f"Translate English to French: {question} ->"
few_shot = "Translate English to French:\n"
few_shot += "".join(f"{en} -> {fr}\n" for en, fr in examples)
few_shot += f"{question} ->"

print("ZERO-SHOT PROMPT:\n" + zero_shot + "\n")
print("FEW-SHOT PROMPT:\n" + few_shot)

## 13. InstructGPT and ChatGPT: Learning from Human Feedback (2022)

**Example:** Where did the cat sit? → **On the mat.**

In [ ]:
import numpy as np

# Question: "Where did the cat sit?"
answers = ["On the mat.",                      # helpful
           "the cat sat on the mat the cat sat", # repeats itself
           "Why do you care?"]                 # rude
prob = np.array([0.2, 0.5, 0.3])              # base model's habits (before RLHF)

# Reward model = trained on human ratings. Here: simple hand rules.
def reward(text):
    r = 0.0
    if "mat" in text.lower(): r += 1          # answers the question
    if len(text.split()) > 6: r -= 1           # too long / repeats
    if "why" in text.lower(): r -= 1           # rude
    return r

rewards = np.array([reward(a) for a in answers])
print("rewards:", rewards)

# RL step (simplified): push probability toward high-reward answers
for round_ in range(1, 4):
    prob = prob * np.exp(rewards)
    prob /= prob.sum()
    print(f"after round {round_}:", {a[:18]: f"{p:.2f}" for a, p in zip(answers, prob)})

## 14. Retrieval-Augmented Generation and Tool Use (2020-2023)

**Example:** Where does the cat sleep? → **On the blue mat in the kitchen.**

In [ ]:
# RAG: find the most relevant document, then put it in the prompt.
documents = [
    "Our cat Tom sleeps on the blue mat in the kitchen.",
    "The library opens at 9 am on Mondays.",
    "Fish should be fed twice a day.",
]
question = "Where does the cat sleep?"

def overlap(q, doc):                       # count shared words
    q_words = set(q.lower().strip("?").split())
    d_words = set(doc.lower().strip(".").split())
    return len(q_words & d_words)

scores = [overlap(question, d) for d in documents]
best = documents[scores.index(max(scores))]
print("scores:", scores)

prompt = f"Use this information:\n{best}\n\nQuestion: {question}\nAnswer:"
print(prompt)

## 15. Reasoning Models (2024-2025)

**Example:** The cat had 3 fish, got 2 more, then ate 1. How many are left? → **4**

In [ ]:
import random
random.seed(7)

# Problem: "The cat had 3 fish, got 2 more, then ate 1. How many are left?"
# A noisy "model" makes a small slip 30% of the time on each step.
def noisy(x):
    return x + random.choice([-1, 1]) if random.random() < 0.3 else x

def answer_directly():                   # one jump to the answer: big slip chance
    return noisy(noisy(noisy(3 + 2 - 1)))

def answer_step_by_step():               # write steps, check each one
    a = 3                                  # step 1: start with 3
    b = a + 2                              # step 2: got 2 more -> 5
    c = b - 1                              # step 3: ate 1      -> 4
    return noisy(c)                        # only one place left to slip

def vote(fn, n=5):                       # think several times, take majority
    results = [fn() for _ in range(n)]
    return max(set(results), key=results.count)

trials = 1000
for name, fn in [("direct", answer_directly),
                 ("step-by-step", answer_step_by_step),
                 ("step-by-step + vote", lambda: vote(answer_step_by_step))]:
    correct = sum(fn() == 4 for _ in range(trials))
    print(f"{name:22s} accuracy: {correct / trials:.0%}")

## 16. Agentic LLMs: Today (2025-2026)

**Example:** 23 tins of cat food cost 47 rupees each, plus 5 delivery. Total? → **1086 rupees**

In [ ]:
# Agent loop: think -> use a tool -> look at result -> answer.
def calculator(expression):
    return eval(expression, {"__builtins__": {}})   # demo only

def fake_llm(history):
    # A real LLM would decide this itself; we hard-code its choices.
    if not any(h.startswith("TOOL RESULT") for h in history):
        return "CALL calculator: 23 * 47 + 5"
    return "ANSWER: The cat food costs " + history[-1].split(": ")[1] + " rupees."

history = ["USER: 23 tins of cat food cost 47 rupees each, plus 5 delivery. Total?"]
for step in range(5):
    action = fake_llm(history)
    print("MODEL:", action)
    if action.startswith("CALL calculator:"):
        result = calculator(action.split(": ", 1)[1])
        history.append(f"TOOL RESULT: {result}")
        print("TOOL :", result)
    else:
        break